# DA3 几何工具测试

加载一次 DA3 模型后复用，对 ScanNet 均匀采样帧进行推理，检查深度、点云、BEV、3DGS 和 NVS 输出。

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

PROJECT_ROOT = Path('/home/zhouruofan/Training-Free/SpatialMem')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tools.visual_generation.da3_geometry import DA3GeometryTool


In [ ]:
# 配置：模型路径、数据路径、统一输出路径、设备
MODEL_DIR = Path('/home/zhouruofan/Training-Free/tool_model/DA3NESTED-GIANT-LARGE-1.1')
DA3_REPO = Path('/home/zhouruofan/Training-Free/tool_model/depth-anything-3')
IMAGE_DIR = Path('/home/zhouruofan/datasets/spar/scannet/images/scene0000_00/image_color')

# 统一输出根目录，各工具自动保存到子目录
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
DA3_OUTPUT_DIR = OUTPUT_ROOT / 'da3'
SAM3_OUTPUT_DIR = OUTPUT_ROOT / 'sam3'
SPATIAL_OUTPUT_DIR = OUTPUT_ROOT / 'spatial'

DEVICE = 'cuda:0'
PROCESS_RES = 504

torch.cuda.set_device(DEVICE)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device count:', torch.cuda.device_count())
    print('default device:', torch.cuda.current_device())

In [ ]:
# 加载 DA3 模型权重（只需加载一次，后续复用）
da3_tool = DA3GeometryTool(
    model_dir=MODEL_DIR,
    da3_repo=DA3_REPO,
    device=DEVICE,
)

In [ ]:
# 从数据集中均匀采样帧，每隔 frame_stride 帧取一帧
IMAGE_DIR = Path('/home/zhouruofan/datasets/spar/scannet/images/scene0000_00/image_color')

all_images = sorted(IMAGE_DIR.glob('*.jpg'), key=lambda p: int(p.stem))
frame_stride = 100
num_frames = 10

indices = np.arange(0, len(all_images), frame_stride, dtype=int)[:num_frames]
assert len(indices) == num_frames, (
    f'Need {num_frames} frames with stride {frame_stride}, '
    f'but only found {len(indices)} from {len(all_images)} images.'
)
frame_paths = [str(all_images[i]) for i in indices]

print('总图片数:', len(all_images))
print('采样步长:', frame_stride)
print('选取索引:', indices.tolist())
print('选取帧 ID:', [Path(p).stem for p in frame_paths])
frame_paths

In [ ]:
# 运行 DA3 推理并保存到 outputs/da3/
result = da3_tool.run(
    frame_paths=frame_paths,
    output_dir=DA3_OUTPUT_DIR,
    process_res=PROCESS_RES,
    process_res_method='upper_bound_resize',
    use_ray_pose=False,
    ref_view_strategy='saddle_balanced',
    save_depth_vis=True,
)

print('保存路径:', DA3_OUTPUT_DIR)
print('depth:', result['depth'].shape)
print('conf:', None if result['conf'] is None else result['conf'].shape)
print('extrinsics:', None if result['extrinsics'] is None else result['extrinsics'].shape)
print('intrinsics:', None if result['intrinsics'] is None else result['intrinsics'].shape)
print('is_metric:', result['is_metric'])
print('scale_factor:', result['scale_factor'])

In [ ]:
# 可视化第一帧：原始图像、预测深度图、相机外参和内参
first_image = Image.open(frame_paths[0]).convert('RGB')
first_depth = result['depth'][0]
first_extrinsic = result['extrinsics'][0]
first_intrinsic = result['intrinsics'][0]

valid = np.isfinite(first_depth) & (first_depth > 0)
lo, hi = np.percentile(first_depth[valid], [2, 98]) if valid.any() else (0.0, 1.0)
first_depth_vis = np.clip((first_depth - lo) / max(hi - lo, 1e-6), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(first_image)
axes[0].set_title(f'Original frame {Path(frame_paths[0]).stem}')
axes[0].axis('off')

im = axes[1].imshow(first_depth_vis, cmap='inferno')
axes[1].set_title('DA3 depth (normalized)')
axes[1].axis('off')
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()

np.set_printoptions(precision=4, suppress=True)
print('第一帧外参 (w2c):')
print(first_extrinsic)
print('\n第一帧内参:')
print(first_intrinsic)

## 从 DA3 输出生成点云

利用深度图和相机参数反投影生成世界坐标系下的彩色点云，供后续几何计算使用。

In [ ]:
# 深度反投影生成世界坐标系点云
from tools.visual_generation.da3_geometry import prediction_to_point_cloud

prediction = result['prediction']
points, colors = prediction_to_point_cloud(prediction, use_conf=False)

print('点数:', points.shape, points.dtype)
print('颜色:', colors.shape, colors.dtype)
print('xyz 最小值:', points.min(axis=0))
print('xyz 最大值:', points.max(axis=0))

In [ ]:
# 将点云保存为 NPZ 格式到 outputs/da3/
pointcloud_npz = DA3_OUTPUT_DIR / 'point_cloud.npz'
np.savez_compressed(
    pointcloud_npz,
    points=points.astype(np.float32),
    colors=colors.astype(np.uint8),
)
print('已保存:', pointcloud_npz)

In [ ]:
# 可选：导出 PLY 格式到 outputs/da3/
try:
    import open3d as o3d

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64) / 255.0)

    pointcloud_ply = DA3_OUTPUT_DIR / 'point_cloud.ply'
    o3d.io.write_point_cloud(str(pointcloud_ply), pcd)
    print('已保存:', pointcloud_ply)
except Exception as exc:
    print('PLY 导出跳过:', repr(exc))

In [ ]:
# 点云 3D 预览（降采样以保证绘图流畅）
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

max_preview = 60000
if len(points) > max_preview:
    rng = np.random.default_rng(0)
    preview_idx = rng.choice(len(points), size=max_preview, replace=False)
else:
    preview_idx = np.arange(len(points))

preview_points = points[preview_idx]
preview_colors = colors[preview_idx].astype(np.float32) / 255.0

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(
    preview_points[:, 0],
    preview_points[:, 1],
    preview_points[:, 2],
    c=preview_colors,
    s=0.2,
)
ax.set_title('DA3 point cloud preview')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

# 等比例缩放坐标轴
center = preview_points.mean(axis=0)
radius = np.percentile(np.linalg.norm(preview_points - center, axis=1), 95)
ax.set_xlim(center[0] - radius, center[0] + radius)
ax.set_ylim(center[1] - radius, center[1] + radius)
ax.set_zlim(center[2] - radius, center[2] + radius)
plt.show()

In [ ]:
# 从点云生成鸟瞰图（BEV），保存到 outputs/da3/
import matplotlib.pyplot as plt

bev_result = da3_tool.make_bev(
    result['prediction'],
    output_dir=DA3_OUTPUT_DIR,
    horizontal_axes=(0, 2),
    target_max_side_px=1200,
    use_conf=False,
    rotate_tall_display=True,
    pad_to_square_display=True,
)

bev_display = bev_result['bev_display']

print('有效点数:', len(bev_result['points']))
print('显示 BEV 尺寸 HxW:', bev_display.shape[:2])

plt.figure(figsize=(10, 10))
plt.imshow(bev_display, aspect='equal')
plt.title('BEV from DA3 back-projected point cloud')
plt.axis('off')
plt.tight_layout(pad=0)
plt.show()

In [ ]:
# 运行 DA3 3DGS 推理并导出高斯 PLY 到 outputs/da3/
prediction_gs = da3_tool.infer(
    frame_paths,
    process_res=PROCESS_RES,
    infer_gs=True,
)

assert prediction_gs.gaussians is not None, '未生成高斯，请检查 MODEL_DIR 是否为 DA3-Giant。'

ply_path = da3_tool.export_gaussian_ply(prediction_gs, output_dir=DA3_OUTPUT_DIR)

print('高斯 PLY:', ply_path)
print('gaussian means:', tuple(prediction_gs.gaussians.means.shape))

In [ ]:
# 验证 3DGS 渲染质量：从原始相机 0 渲染并与输入帧对比

import numpy as np
import matplotlib.pyplot as plt

assert prediction_gs.gaussians is not None, '请先运行 3DGS 推理 cell。'

step1 = da3_tool.render_nvs(
    prediction_gs,
    frame_index=0,
    movement='yaw_left',
    angle_deg=0.0,
)
orig_render = step1['image']

orig_image = prediction_gs.processed_images[0]
if orig_image.max() > 1:
    orig_image = orig_image.astype(np.float32) / 255.0

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(orig_image)
axes[0].set_title('Processed input frame 0')
axes[0].axis('off')
axes[1].imshow(orig_render)
axes[1].set_title('3DGS render from original camera 0')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 新视角合成（NVS）：从第一帧相机向左偏航 30 度渲染
import matplotlib.pyplot as plt

nvs_result = da3_tool.render_nvs(
    prediction_gs,
    frame_index=0,
    movement='yaw_left',
    angle_deg=30.0,
    output_dir=DA3_OUTPUT_DIR,
    show_reference=True,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(nvs_result['reference_image'])
axes[0].set_title('3DGS render: original camera')
axes[0].axis('off')
axes[1].imshow(nvs_result['image'])
axes[1].set_title('3DGS NVS: yaw left 30 deg')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## SAM3 逐帧分割

加载 SAM3 图像模型，对每帧独立做文本 prompt 检测。后续通过 3D 聚类关联跨帧实例。

In [ ]:
# 加载 SAM3 图像模型（只需加载一次，后续复用）
from tools.visual_generation.sam3_segmentation import SAM3SegmentationTool, visualize_segmentation

sam3_tool = SAM3SegmentationTool(device=DEVICE, confidence_threshold=0.3)

In [ ]:
# 逐帧文本检测，结果保存到 outputs/sam3/
TEXT_PROMPT = "chair"

seg_results = sam3_tool.segment_frames(
    frame_paths, text_prompt=TEXT_PROMPT, output_dir=SAM3_OUTPUT_DIR,
)

for i, res in enumerate(seg_results):
    frame_id = Path(frame_paths[i]).stem
    n = len(res["scores"])
    scores_str = f'  scores={res["scores"].round(2).tolist()}' if n > 0 else ''
    print(f'frame {frame_id}: {n} "{TEXT_PROMPT}"{scores_str}')

In [ ]:
# 可视化每帧的分割结果
visualize_segmentation(frame_paths, seg_results, TEXT_PROMPT, output_dir=SAM3_OUTPUT_DIR)

## 实例 3D 位置计算

利用 DA3 深度图 + 相机位姿，将 SAM3 的 2D 分割 mask 反投影到世界坐标系，得到每个实例的 3D 质心位置。

In [ ]:
# 逐帧 3D 反投影 + 3D 聚类合并同一实例，保存到 outputs/spatial/
from tools.code_execution.instance_3d_localization import compute_instance_3d_positions, print_instance_3d_summary

results_3d = compute_instance_3d_positions(
    seg_results=seg_results,
    depth=result['depth'],
    intrinsics=result['intrinsics'],
    extrinsics=result['extrinsics'],
    distance_threshold=0.5,
    output_dir=SPATIAL_OUTPUT_DIR,
)

print_instance_3d_summary(results_3d, TEXT_PROMPT)

In [ ]:
# 实例计数（从 3D 聚类结果中获取）
from tools.code_execution.instance_counting import count_unique_instances, print_counting_summary

counting = count_unique_instances(results_3d)
print_counting_summary(counting, TEXT_PROMPT)

## 距离计算 & 方位计算测试

In [ ]:
from tools.code_execution.distance_computation import (
    distance_object_to_object, distance_object_to_camera, get_camera_position,
)
from tools.code_execution.direction_computation import compute_relative_direction

extrinsics = result['extrinsics']
chair_0 = results_3d['merged_positions'][0]
chair_1 = results_3d['merged_positions'][1]

# --- 距离计算 ---

d_chairs = distance_object_to_object(chair_0, chair_1)
print(f'椅子{results_3d["obj_id_list"][0]} 与 椅子{results_3d["obj_id_list"][1]} 的距离: {d_chairs:.3f}')

d_cam0 = distance_object_to_camera(chair_0, extrinsics, frame_index=0)
d_cam1 = distance_object_to_camera(chair_1, extrinsics, frame_index=0)
print(f'椅子{results_3d["obj_id_list"][0]} 到 帧0 相机距离: {d_cam0:.3f}')
print(f'椅子{results_3d["obj_id_list"][1]} 到 帧0 相机距离: {d_cam1:.3f}')
print(f'→ 离帧0相机更近的是: 椅子{results_3d["obj_id_list"][0] if d_cam0 < d_cam1 else results_3d["obj_id_list"][1]}')

print()

# --- 方位计算 ---

dir_result = compute_relative_direction(
    viewpoint=0, reference=chair_0, target=chair_1, extrinsics=extrinsics,
)
print(f'从帧0相机看，椅子{results_3d["obj_id_list"][1]} 在 椅子{results_3d["obj_id_list"][0]} 的 {dir_result["direction"]} 方向')
print(f'  左右偏角: {dir_result["lr_angle_deg"]:.1f}°  前后偏角: {dir_result["fb_angle_deg"]:.1f}°')

## 场景尺寸计算

基于 metric 矫正后的点云，过滤离群点后计算场景包围盒尺寸。

In [ ]:
# 计算场景尺寸，使用 DA3 置信度过滤低质量点 + 空间裁剪离群点
from tools.code_execution.scene_size_computation import compute_scene_size, print_scene_size

scene_size = compute_scene_size(
    points=points,
    conf=result['conf'].reshape(-1) if result['conf'] is not None else None,
    conf_percentile=10.0,
    spatial_percentile=2.0,
    output_dir=SPATIAL_OUTPUT_DIR,
)

print_scene_size(scene_size)